In [4]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)

In [6]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [7]:
def create_joke(state: JokeState):
    prompt = f"Generate a joke on the topic of {state['topic']}"

    response = model.invoke(prompt).content

    return {
        'joke':response
    }

def explain_joke(state: JokeState):
    prompt = f"Explain the following joke:\n {state['joke']}"

    response = model.invoke(prompt).content

    return {
        'explanation':response
    }

In [8]:
graph = StateGraph(JokeState)

graph.add_node("create_joke", create_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "create_joke")
graph.add_edge("create_joke", "explain_joke")
graph.add_edge("explain_joke", END)

checkpoint = InMemorySaver()

workflow = graph.compile(checkpointer = checkpoint)

In [9]:
config1 = {
    "configurable":{'thread_id':"1"}
}
workflow.invoke({'topic': 'technology'},config=config1)

{'topic': 'technology',
 'joke': 'Why did the computer go to therapy?  \n\nIt had too many *bytes* of unresolved issues and kept crashing its own “software” relationships!',
 'explanation': '**The joke**  \n\n> *Why did the computer go to therapy?*  \n> *It had too many **bytes** of unresolved issues and kept crashing its own “software” relationships!*\n\n---\n\n## 1. The surface‑level “story”\n\nA computer is treated like a person who needs therapy.  \nIn the punch‑line we hear two computer‑related terms—**bytes** and **crashing**—used as if they were emotional problems:\n\n* “too many **bytes** of unresolved issues” → the computer is “full of problems”  \n* “kept **crashing** its own ‘software’ relationships” → the computer repeatedly “breaks up” with its programs.\n\nThe absurd image of a computer on a therapist’s couch, spilling its “feelings,” creates the humor.\n\n---\n\n## 2. Why each wordplay works\n\n| Computer term | Literal meaning (tech) | Figurative meaning (psychology) | 

In [12]:
config2 = {
    "configurable":{'thread_id':"2"}
}
workflow.invoke({'topic': 'humans'},config=config2)

{'topic': 'humans',
 'joke': 'Why did the human bring a ladder to the bar?  \n\nBecause they heard the drinks were on the “top shelf” of evolution—and they still haven’t figured out how to reach the “spirit” level without a step‑up!',
 'explanation': '**TL;DR:**  \nThe joke is a multi‑layered word‑play that mixes three different meanings of “top shelf” and “spirit level” with the visual gag of someone hauling a ladder into a bar. The humor comes from the clash between a literal, clumsy solution (a ladder) and the figurative, “high‑brow” ways we talk about premium drinks, evolution, and measuring flatness.\n\n---\n\n## 1. The set‑up: “Why did the human bring a ladder to the bar?”\n\n- **Visual gag** – Imagine a person walking into a tavern, ladder in hand, looking for a drink. It’s absurd because you never need a ladder to order a cocktail.\n- **Expectation subversion** – We anticipate a punch‑line about drunkenness, a bar‑related idiom, or a simple literal reason (“the bar was too high

In [13]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'humans', 'joke': 'Why did the human bring a ladder to the bar?  \n\nBecause they heard the drinks were on the “top shelf” of evolution—and they still haven’t figured out how to reach the “spirit” level without a step‑up!', 'explanation': '**TL;DR:**  \nThe joke is a multi‑layered word‑play that mixes three different meanings of “top shelf” and “spirit level” with the visual gag of someone hauling a ladder into a bar. The humor comes from the clash between a literal, clumsy solution (a ladder) and the figurative, “high‑brow” ways we talk about premium drinks, evolution, and measuring flatness.\n\n---\n\n## 1. The set‑up: “Why did the human bring a ladder to the bar?”\n\n- **Visual gag** – Imagine a person walking into a tavern, ladder in hand, looking for a drink. It’s absurd because you never need a ladder to order a cocktail.\n- **Expectation subversion** – We anticipate a punch‑line about drunkenness, a bar‑related idiom, or a simple literal reason (“t

In [14]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'technology', 'joke': 'Why did the computer go to therapy?  \n\nIt had too many *bytes* of unresolved issues and kept crashing its own “software” relationships!', 'explanation': '**The joke**  \n\n> *Why did the computer go to therapy?*  \n> *It had too many **bytes** of unresolved issues and kept crashing its own “software” relationships!*\n\n---\n\n## 1. The surface‑level “story”\n\nA computer is treated like a person who needs therapy.  \nIn the punch‑line we hear two computer‑related terms—**bytes** and **crashing**—used as if they were emotional problems:\n\n* “too many **bytes** of unresolved issues” → the computer is “full of problems”  \n* “kept **crashing** its own ‘software’ relationships” → the computer repeatedly “breaks up” with its programs.\n\nThe absurd image of a computer on a therapist’s couch, spilling its “feelings,” creates the humor.\n\n---\n\n## 2. Why each wordplay works\n\n| Computer term | Literal meaning (tech) | Figurative mea